# Build Classified ChatDev Scored Dataset

Score every summarized JSON under `data/classified_chatdev_summarized` with `chatdev.analyzer.logprob_consistency`, and write the results under `data/classified_chatdev_scored` with the same label-directory layout. Duplicate samples are detected by file name: each unique summarized filename is scored once, then copied to other matching label directories.

In [1]:
from pathlib import Path
import json
import shutil
import sys
import time

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INPUT_ROOT = PROJECT_ROOT / "data" / "classified_chatdev_summarized"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "classified_chatdev_scored"
DATASET_JSON = PROJECT_ROOT / "chatdev_dataset.json"
TRAJECTORY_DIR = PROJECT_ROOT / "data" / "classified_chatdev_playbook" / "trajectory"
MANIFEST_PATH = OUTPUT_ROOT / "_manifest.json"

# Tune these before running if needed.
MODEL = "gpt-4o-mini"
TOP_LOGPROBS = 5
OVERWRITE = False
CONTINUE_ON_ERROR = True
MAX_FILES = None  # set to an integer for a small test run

INPUT_ROOT, OUTPUT_ROOT, DATASET_JSON, TRAJECTORY_DIR

(WindowsPath('d:/Works/code/winter-like-ai/ChatDev/data/classified_chatdev_summarized'),
 WindowsPath('d:/Works/code/winter-like-ai/ChatDev/data/classified_chatdev_scored'),
 WindowsPath('d:/Works/code/winter-like-ai/ChatDev/chatdev_dataset.json'),
 WindowsPath('d:/Works/code/winter-like-ai/ChatDev/data/classified_chatdev_playbook/trajectory'))

In [2]:
from chatdev.analyzer.logprob_consistency import (
    LogprobConsistencyScorer,
    infer_user_task_for_path,
    load_user_task_map,
)


def scored_output_path(src_path: Path) -> Path:
    rel = src_path.relative_to(INPUT_ROOT)
    name = rel.name
    if name.endswith("_summarized.json"):
        name = name[:-len("_summarized.json")] + "_scored.json"
    else:
        name = rel.stem + "_scored.json"
    return OUTPUT_ROOT / rel.parent / name


def write_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def scored_output_has_user_task(path: Path) -> bool:
    if not path.exists():
        return False
    try:
        with path.open("r", encoding="utf-8") as f:
            payload = json.load(f)
    except Exception:
        return False
    for interactions in payload.values():
        if not isinstance(interactions, list):
            continue
        for entry in interactions:
            if isinstance(entry, dict):
                return bool(entry.get("user_task"))
    return False


def discover_summarized_paths() -> list[Path]:
    paths = sorted(INPUT_ROOT.rglob("*_summarized.json"))
    paths = [p for p in paths if p.is_file()]
    if MAX_FILES is not None:
        paths = paths[:MAX_FILES]
    return paths


def canonical_sort_key(path: Path):
    rel = path.relative_to(INPUT_ROOT)
    return (0 if rel.parts[0] == "trajectory" else 1, str(rel))


def group_by_filename(paths: list[Path]) -> dict[str, list[Path]]:
    groups = {}
    for path in paths:
        groups.setdefault(path.name, []).append(path)
    return {name: sorted(items, key=canonical_sort_key) for name, items in sorted(groups.items())}


all_paths = discover_summarized_paths()
filename_groups = group_by_filename(all_paths)
user_task_map = load_user_task_map(dataset_path=str(DATASET_JSON), trajectory_dir=str(TRAJECTORY_DIR))

print(f"Input exists: {INPUT_ROOT.exists()} -> {INPUT_ROOT}")
print(f"Found summarized paths: {len(all_paths)}")
print(f"Unique summarized filenames: {len(filename_groups)}")
print(f"User demand mappings: {len(user_task_map)}")

Input exists: True -> d:\Works\code\winter-like-ai\ChatDev\data\classified_chatdev_summarized
Found summarized paths: 448
Unique summarized filenames: 130
User demand mappings: 390


In [3]:
# Build an initial filename map from already completed scored outputs.
# This supports resume without repeating API calls.
filename_to_scored = {}
existing_pairs = 0

for src_path in discover_summarized_paths():
    dst_path = scored_output_path(src_path)
    if dst_path.exists() and scored_output_has_user_task(dst_path):
        filename_to_scored.setdefault(src_path.name, dst_path)
        existing_pairs += 1

print(f"Existing valid scored outputs found: {existing_pairs}")
print(f"Unique filenames already scored: {len(filename_to_scored)}")

Existing valid scored outputs found: 0
Unique filenames already scored: 0


In [4]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
scorer = LogprobConsistencyScorer(model=MODEL, top_logprobs=TOP_LOGPROBS)

manifest = {
    "input_root": str(INPUT_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "dataset_json": str(DATASET_JSON),
    "trajectory_dir": str(TRAJECTORY_DIR),
    "model": MODEL,
    "top_logprobs": TOP_LOGPROBS,
    "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "overwrite": OVERWRITE,
    "records": [],
}

all_paths = discover_summarized_paths()
filename_groups = group_by_filename(all_paths)
print(f"Processing {len(filename_groups)} unique summarized filenames across {len(all_paths)} paths")

for index, (filename, same_name_paths) in enumerate(filename_groups.items(), start=1):
    canonical_src = same_name_paths[0]
    canonical_dst = scored_output_path(canonical_src)
    canonical_rel = canonical_src.relative_to(INPUT_ROOT)
    user_task = infer_user_task_for_path(str(canonical_src), user_task_map=user_task_map)
    record = {
        "index": index,
        "filename": filename,
        "canonical_source": str(canonical_rel),
        "canonical_output": str(canonical_dst.relative_to(OUTPUT_ROOT)),
        "path_count": len(same_name_paths),
        "paths": [str(path.relative_to(INPUT_ROOT)) for path in same_name_paths],
        "has_user_task": bool(user_task),
    }

    try:
        if canonical_dst.exists() and not OVERWRITE and scored_output_has_user_task(canonical_dst):
            record["status"] = "skipped_existing"
            filename_to_scored.setdefault(filename, canonical_dst)
            print(f"[{index}/{len(filename_groups)}] SKIP existing {canonical_rel}")
        elif canonical_dst.exists() and not OVERWRITE and not scored_output_has_user_task(canonical_dst):
            print(f"[{index}/{len(filename_groups)}] RESCORE missing user_task {canonical_rel}")
            scorer.score_summarized_json(
                input_path=str(canonical_src),
                output_path=str(canonical_dst),
                user_task=user_task,
                user_task_map=user_task_map,
                verbose=False,
            )
            filename_to_scored[filename] = canonical_dst
            record["status"] = "rescored_missing_user_task"
        elif filename in filename_to_scored and filename_to_scored[filename].exists():
            canonical_dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(filename_to_scored[filename], canonical_dst)
            record["status"] = "copied_filename_duplicate"
            record["duplicate_of"] = str(filename_to_scored[filename].relative_to(OUTPUT_ROOT))
            print(f"[{index}/{len(filename_groups)}] COPY filename duplicate {canonical_rel}")
        else:
            print(f"[{index}/{len(filename_groups)}] SCORE {canonical_rel}")
            scorer.score_summarized_json(
                input_path=str(canonical_src),
                output_path=str(canonical_dst),
                user_task=user_task,
                user_task_map=user_task_map,
                verbose=False,
            )
            filename_to_scored[filename] = canonical_dst
            record["status"] = "scored"

        copied_outputs = []
        source_scored = filename_to_scored.get(filename, canonical_dst)
        for duplicate_src in same_name_paths:
            duplicate_dst = scored_output_path(duplicate_src)
            if duplicate_dst == source_scored:
                continue
            if duplicate_dst.exists() and not OVERWRITE and scored_output_has_user_task(duplicate_dst):
                continue
            duplicate_dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_scored, duplicate_dst)
            copied_outputs.append(str(duplicate_dst.relative_to(OUTPUT_ROOT)))
        record["copied_outputs"] = copied_outputs
    except Exception as exc:
        record["status"] = "error"
        record["error"] = repr(exc)
        print(f"[{index}/{len(filename_groups)}] ERROR {canonical_rel}: {exc}")
        if not CONTINUE_ON_ERROR:
            manifest["records"].append(record)
            write_json(MANIFEST_PATH, manifest)
            raise

    manifest["records"].append(record)
    if index % 5 == 0:
        write_json(MANIFEST_PATH, manifest)

manifest["finished_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
manifest["scorer_stats"] = scorer.stats
write_json(MANIFEST_PATH, manifest)

print("Done.")
print(json.dumps({
    "records": len(manifest["records"]),
    "statuses": {s: sum(1 for r in manifest["records"] if r.get("status") == s) for s in sorted({r.get("status") for r in manifest["records"]})},
    "scorer_stats": scorer.stats,
    "manifest": str(MANIFEST_PATH),
}, ensure_ascii=False, indent=2))

Processing 130 unique summarized filenames across 448 paths
[1/130] SCORE trajectory\ChatDev_ProgramDev2_GPT4o_0_summarized.json
[2/130] SCORE trajectory\ChatDev_ProgramDev2_GPT4o_10_summarized.json
[3/130] SCORE trajectory\ChatDev_ProgramDev2_GPT4o_11_summarized.json
[4/130] SCORE trajectory\ChatDev_ProgramDev2_GPT4o_12_summarized.json
[5/130] SCORE trajectory\ChatDev_ProgramDev2_GPT4o_13_summarized.json
[6/130] SCORE trajectory\ChatDev_ProgramDev2_GPT4o_14_summarized.json
[7/130] SCORE trajectory\ChatDev_ProgramDev2_GPT4o_15_summarized.json
[8/130] SCORE trajectory\ChatDev_ProgramDev2_GPT4o_16_summarized.json
[9/130] SCORE trajectory\ChatDev_ProgramDev2_GPT4o_17_summarized.json
[10/130] SCORE trajectory\ChatDev_ProgramDev2_GPT4o_18_summarized.json
[11/130] SCORE trajectory\ChatDev_ProgramDev2_GPT4o_19_summarized.json
[12/130] SCORE trajectory\ChatDev_ProgramDev2_GPT4o_1_summarized.json
[13/130] SCORE trajectory\ChatDev_ProgramDev2_GPT4o_20_summarized.json
[14/130] SCORE trajectory\Ch

In [5]:
# Quick structural check: every summarized input should have one scored output.
missing = []
missing_user_task = []
for src_path in discover_summarized_paths():
    dst_path = scored_output_path(src_path)
    if not dst_path.exists():
        missing.append(str(src_path.relative_to(INPUT_ROOT)))
    elif not scored_output_has_user_task(dst_path):
        missing_user_task.append(str(dst_path.relative_to(OUTPUT_ROOT)))

print(f"Missing outputs: {len(missing)}")
print(f"Scored outputs missing user_task: {len(missing_user_task)}")
if missing[:20]:
    print(json.dumps(missing[:20], ensure_ascii=False, indent=2))
if missing_user_task[:20]:
    print(json.dumps(missing_user_task[:20], ensure_ascii=False, indent=2))

# Show one sample scored output path and a compact preview.
sample_outputs = sorted(OUTPUT_ROOT.rglob("*_scored.json"))
if sample_outputs:
    sample = sample_outputs[0]
    with sample.open("r", encoding="utf-8") as f:
        payload = json.load(f)
    first_role = next(iter(payload))
    first_entry = payload[first_role][0]
    preview = {
        "sample": str(sample.relative_to(OUTPUT_ROOT)),
        "role": first_role,
        "phase": first_entry.get("phase"),
        "turn": first_entry.get("turn"),
        "has_user_task": bool(first_entry.get("user_task")),
        "user_task_prefix": (first_entry.get("user_task") or "")[:120],
        "consistency_score_mean": first_entry.get("consistency_score_mean"),
        "num_output_scores": len(first_entry.get("output_consistency_scores", [])),
    }
    print(json.dumps(preview, ensure_ascii=False, indent=2))

Missing outputs: 0
Scored outputs missing user_task: 448
[
  "0.0\\ChatDev_ProgramDev2_GPT4o_0_scored.json",
  "0.0\\ChatDev_ProgramDev2_GPT4o_13_scored.json",
  "0.0\\ChatDev_ProgramDev2_GPT4o_14_scored.json",
  "0.0\\ChatDev_ProgramDev2_GPT4o_16_scored.json",
  "0.0\\ChatDev_ProgramDev2_GPT4o_1_scored.json",
  "0.0\\ChatDev_ProgramDev2_GPT4o_20_scored.json",
  "0.0\\ChatDev_ProgramDev2_GPT4o_22_scored.json",
  "0.0\\ChatDev_ProgramDev2_GPT4o_23_scored.json",
  "0.0\\ChatDev_ProgramDev2_GPT4o_24_scored.json",
  "0.0\\ChatDev_ProgramDev2_GPT4o_25_scored.json",
  "0.0\\ChatDev_ProgramDev2_GPT4o_26_scored.json",
  "0.0\\ChatDev_ProgramDev2_GPT4o_28_scored.json",
  "0.0\\ChatDev_ProgramDev2_GPT4o_29_scored.json",
  "0.0\\ChatDev_ProgramDev2_GPT4o_2_scored.json",
  "0.0\\ChatDev_ProgramDev2_GPT4o_34_scored.json",
  "0.0\\ChatDev_ProgramDev2_GPT4o_4_scored.json",
  "0.0\\ChatDev_ProgramDev2_GPT4o_50_scored.json",
  "0.0\\ChatDev_ProgramDev2_GPT4o_74_scored.json",
  "0.0\\ChatDev_ProgramDev2